FigS3_F CA3

In [ ]:
# Previous_Rv1/sleep/figure_scripts/FigS3_G_F_CA3_CA2_pyr_model_surpport_fig.ipynb, run by ../FigS6_AF_swr_controls.ipynb (final_figs.run_rv1_capture) on
# non-overlapping windows (per session, all states in time order, a window kept only if its centre is at least
# one window length after the last kept one). Thinned or rebuilt tables: CSV; the others: RV1_CSV.
import os, glob, pickle, re, sys   # standard imports
from pathlib import Path
sys.path.insert(0, str(Path('..') / '..' / '..' / '..' / 'methods'))   # the criticality library
import pandas as pd
import numpy as np
CSV = Path('..') / '..' / 'csv' / 'FigS6_AF_swr_controls' / 'rv1_FigS3_G_F_CA3_CA2_pyr_model_surpport_fig'    # thinned / rebuilt tables
RV1_CSV = Path('..') / '..' / '..' / '..' / 'Previous_Rv1' / 'sleep' / 'csv' / 'FigS3_G_F_CA3_CA2_pyr_model_surpport_fig'   # the others
OUT = Path('..') / '..' / 'figure_output'
OUT.mkdir(exist_ok=True)


def read_csv(name, **kw):
    """A table of this notebook (round_trip: the floats exactly as written)."""
    return pd.read_csv(CSV / name if (CSV / name).exists() else RV1_CSV / name, float_precision='round_trip', **kw)

In [ ]:
import pickle
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob

# ==========================================
# CONFIGURATION
# ==========================================
save_dir = r"D:\Documents\GitHub\Sleep_criticaltiy\data\rats\barr_CA2_CA3"

BINNING_METHOD = 'equal_width'   # 'equal_count' (forced sextile) or 'equal_width'
n_tiles = 6

fr_columns = [
    'CA3_E_FR_per_unit', 'CA3_I_FR_per_unit', 'CA3_All_FR_per_unit'
]

# ==========================================
# LOAD DATA
# ==========================================
# ---- data (../csv) ----
# one table per session, in the original file order (column _file; _name = the session's file)
_t = read_csv('rat_CA2_CA3_firing_by_window.csv')
all_dfs = [g.drop(columns=['_file', '_name']).reset_index(drop=True) for _, g in _t.groupby('_file', sort=True)]
pkl_files = [g['_name'].iloc[0] for _, g in _t.groupby('_file', sort=True)]
# ---- the same from the recordings, with analysis_scripts/: ----
# pkl_files = glob(os.path.join(save_dir, "*.pkl"))
# print(f"Found {len(pkl_files)} pkl files")
# print("="*60)
# 
# all_dfs = []
# for pkl_file in pkl_files:
#     with open(pkl_file, 'rb') as f:
#         df = pickle.load(f)
#     filename = os.path.basename(pkl_file)
#     print(f"Loaded: {filename} - Shape: {df.shape}")
#     all_dfs.append(df)

print("\n" + "="*60)
print(f"Total files loaded: {len(all_dfs)}")
print("="*60)

# ==========================================
# HELPERS
# ==========================================

def zscore_within_vector(y):
    """Z-score a 1D array; returns NaNs if std=0 or too few points."""
    y = np.asarray(y, float)
    mu = np.nanmean(y)
    sd = np.nanstd(y)
    if not np.isfinite(sd) or sd == 0:
        return np.full_like(y, np.nan, dtype=float)
    return (y - mu) / sd

def compute_binned_stats_zscore(df, fr_column, n_tiles=6, method='equal_width'):
    """
    For ONE FILE and ONE fr_column:
      1) keep ripple_rate > 0
      2) drop NaNs in x and y
      3) z-score y within this file+region+celltype (i.e., within this column)
      4) compute sextile (equal_count) OR equal_width bins on x
      5) return per-tile means of x and z(y)
    """
    if ('ripple_rate' not in df.columns) or (fr_column not in df.columns):
        return None

    df_filtered = df[(df["state"] == "NREM")].copy()
    if len(df_filtered) == 0:
        return None

    x = df_filtered['ripple_rate'].to_numpy(dtype=float)
    y = df_filtered[fr_column].to_numpy(dtype=float)

    mask = ~(np.isnan(x) | np.isnan(y))
    x_clean = x[mask]
    y_clean = y[mask]
    if len(x_clean) < 3:
        return None

    # --- z-score y within this file+region+celltype (i.e., within this column) ---
    y_z = zscore_within_vector(y_clean)
    # If zscore failed (std==0), bail
    if np.all(np.isnan(y_z)):
        return None

    # --- bin edges from x_clean ---
    if method == 'equal_count':
        # force sextiles by percentiles
        percentiles = np.linspace(0, 100, n_tiles + 1)
        tile_edges = np.percentile(x_clean, percentiles)

        # Important: if barrage_rate has many ties, percentiles can repeat -> empty bins.
        # We'll still handle via NaNs/0 counts below.
    elif method == 'equal_width':
        tile_edges = np.linspace(np.min(x_clean), np.max(x_clean), n_tiles + 1)
    else:
        raise ValueError(f"Unknown method: {method}")

    tile_means_x = []
    tile_means_yz = []
    tile_counts = []

    for i in range(n_tiles):
        if i == n_tiles - 1:
            mask_tile = (x_clean >= tile_edges[i]) & (x_clean <= tile_edges[i+1])
        else:
            mask_tile = (x_clean >= tile_edges[i]) & (x_clean < tile_edges[i+1])

        count = int(np.sum(mask_tile))
        tile_counts.append(count)

        if count > 0:
            tile_means_x.append(np.mean(x_clean[mask_tile]))
            tile_means_yz.append(np.mean(y_z[mask_tile]))
        else:
            tile_means_x.append(np.nan)
            tile_means_yz.append(np.nan)

    return {
        "tile_means_x": np.array(tile_means_x, float),
        "tile_means_y": np.array(tile_means_yz, float),  # z-scored means
        "tile_counts": np.array(tile_counts, int),
        "n_points": int(len(x_clean))
    }

# ==========================================
# COMPUTE PER-FILE STATS THEN POOL
# ==========================================
print("\n" + "="*60)
print(f"Computing Z-scored statistics using method: {BINNING_METHOD.upper()}...")
print("="*60)

pooled_data = {col: [] for col in fr_columns}

for idx, df in enumerate(all_dfs):
    filename = os.path.basename(pkl_files[idx])
    print(f"\nProcessing: {filename}")

    for col in fr_columns:
        stats = compute_binned_stats_zscore(df, col, n_tiles=n_tiles, method=BINNING_METHOD)
        if stats is not None:
            pooled_data[col].append(stats)
            print(f"  {col}: {stats['n_points']} points, tile counts: {stats['tile_counts']}")
        else:
            print(f"  {col}: skipped (missing/empty/zero-std)")

print("\n" + "="*60)
print("Computing pooled statistics (mean ± SEM across files per tile)...")
print("="*60)

pooled_stats = {}

for col in fr_columns:
    if len(pooled_data[col]) == 0:
        print(f"{col}: No data")
        continue

    all_tile_means_x = np.array([s["tile_means_x"] for s in pooled_data[col]], float)  # files x tiles
    all_tile_means_y = np.array([s["tile_means_y"] for s in pooled_data[col]], float)  # files x tiles (z)

    pooled_mean_x = np.nanmean(all_tile_means_x, axis=0)
    pooled_mean_y = np.nanmean(all_tile_means_y, axis=0)

    n_valid = np.sum(~np.isnan(all_tile_means_y), axis=0)
    pooled_std = np.nanstd(all_tile_means_y, axis=0)

    with np.errstate(divide='ignore', invalid='ignore'):
        pooled_sem = pooled_std / np.sqrt(n_valid)

    pooled_stats[col] = {
        "mean_x": pooled_mean_x,
        "mean_y": pooled_mean_y,
        "sem_y": pooled_sem,
        "n_files": len(pooled_data[col]),
        "n_valid": n_valid
    }
    print(f"{col}: pooled from {len(pooled_data[col])} files")

# ==========================================
# PLOTTING (Z-SCORED FR)
# ==========================================
# ==========================================
print("\n" + "="*60)
print("Creating pooled plots (z-scored FR, x = tile)...")
print("="*60)

In [ ]:
import matplotlib as mpl
import scipy.stats as spstats

# Global font: prefer Arial, with safe fallbacks
mpl.rcParams['svg.fonttype'] = 'none' 
mpl.rcParams['text.usetex'] = False
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = [
    'Helvetica Neue', 'Helvetica', 'Arial', 'Liberation Sans', 'DejaVu Sans'
]

# Mathtext to match
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Helvetica Neue'   # or 'Helvetica'
mpl.rcParams['mathtext.it'] = 'Helvetica Neue:italic'
mpl.rcParams['mathtext.bf'] = 'Helvetica Neue:bold'

# (optional) sizes + minus sign
mpl.rcParams['font.size'] = 16
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams['axes.unicode_minus'] = False

# Global sizes (tweak to taste)
mpl.rcParams['font.size'] = 14
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams["axes.linewidth"] = 1.5       # spine width
mpl.rcParams["xtick.major.width"] =1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5
mpl.rcParams["axes.linewidth"] = 1.5       # spine width
mpl.rcParams["xtick.major.width"] =1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5

col_name = "CA3_E_FR_per_unit"
title = "CA3 Excitatory"

fig, ax = plt.subplots(figsize=(6, 5))

st = pooled_stats[col_name]

tile_ids = np.arange(1, n_tiles + 1)
valid = np.isfinite(st["mean_y"])

x_plot = tile_ids[valid]
y_plot = st["mean_y"][valid]
sem_plot = st["sem_y"][valid]

ax.errorbar(
    x_plot, y_plot, yerr=sem_plot,color="tab:red",
    fmt='o-', linewidth=2.5, markersize=10,
    capsize=6, capthick=2.5,label=None
)

ax.set_xlabel("SWRs rate sextile")
ax.set_ylabel("CA3 Pyr mean firing rate (z)")

ax.set_xticks(tile_ids)
ax.set_xticklabels([f"{i}" for i in tile_ids])

ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend()

plt.tight_layout()
save_path=r"../figure_output\S3_F_pooled_CA3_E_FR_per_unit_vs_ripple_tiles.svg"
plt.savefig(save_path, dpi=300)
plt.show()
# weights from SEM (avoid divide-by-zero)
w = 1.0 / np.clip(sem_plot.astype(float), 1e-12, np.inf)**2

# weighted linear fit: y = b0 + b1 x
b1, b0 = np.polyfit(x_plot.astype(float), y_plot.astype(float), deg=1, w=np.sqrt(w))

# get an approximate p-value for slope using weighted regression
# (using effective residual variance)
yhat = b0 + b1 * x_plot
resid = y_plot - yhat
dof = len(x_plot) - 2
s2 = np.sum(w * resid**2) / dof
Sxx = np.sum(w * (x_plot - np.average(x_plot, weights=w))**2)
se_b1 = np.sqrt(s2 / Sxx)
t = b1 / se_b1
p_slope = 2 * spstats.t.sf(np.abs(t), dof)

print(f"Weighted slope = {b1:.3f} ± {se_b1:.3f} (SE), t={t:.2f}, p={p_slope:.3g}, n={len(x_plot)}")
from scipy import stats
import numpy as np

# x_plot, y_plot already defined
r, p = stats.pearsonr(x_plot.astype(float), y_plot.astype(float))
print(f"Pearson r = {r:.3f}, p = {p:.3g}, n = {len(x_plot)}")

# optional: add to title
ax.set_title(f"{title} (Pearson r={r:.2f}, p={p:.2g})")

FigS3_G CA2

In [ ]:
import pickle
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob

# ==========================================
# CONFIGURATION
# ==========================================
save_dir = r"D:\Documents\GitHub\Sleep_criticaltiy\data\rats\barr_CA2_CA3"

BINNING_METHOD = 'equal_width'   # 'equal_count' (forced sextile) or 'equal_width'
n_tiles = 6

fr_columns = [
    'CA2_E_FR_per_unit'
]

# ==========================================
# LOAD DATA
# ==========================================
# ---- data (../csv) ----
# one table per session, in the original file order (column _file; _name = the session's file)
_t = read_csv('rat_CA2_CA3_firing_by_window.csv')
all_dfs = [g.drop(columns=['_file', '_name']).reset_index(drop=True) for _, g in _t.groupby('_file', sort=True)]
pkl_files = [g['_name'].iloc[0] for _, g in _t.groupby('_file', sort=True)]
# ---- the same from the recordings, with analysis_scripts/: ----
# pkl_files = glob(os.path.join(save_dir, "*.pkl"))
# print(f"Found {len(pkl_files)} pkl files")
# print("="*60)
# 
# all_dfs = []
# for pkl_file in pkl_files:
#     with open(pkl_file, 'rb') as f:
#         df = pickle.load(f)
#     filename = os.path.basename(pkl_file)
#     print(f"Loaded: {filename} - Shape: {df.shape}")
#     all_dfs.append(df)

print("\n" + "="*60)
print(f"Total files loaded: {len(all_dfs)}")
print("="*60)

# ==========================================
# HELPERS
# ==========================================

def zscore_within_vector(y):
    """Z-score a 1D array; returns NaNs if std=0 or too few points."""
    y = np.asarray(y, float)
    mu = np.nanmean(y)
    sd = np.nanstd(y)
    if not np.isfinite(sd) or sd == 0:
        return np.full_like(y, np.nan, dtype=float)
    return (y - mu) / sd

def compute_binned_stats_zscore(df, fr_column, n_tiles=6, method='equal_width'):
    """
    For ONE FILE and ONE fr_column:
      1) keep barrage_rate > 0
      2) drop NaNs in x and y
      3) z-score y within this file+region+celltype (i.e., within this column)
      4) compute sextile (equal_count) OR equal_width bins on x
      5) return per-tile means of x and z(y)
    """
    if ('barrage_rate' not in df.columns) or (fr_column not in df.columns):
        return None

    df_filtered = df[(df["state"] == "NREM")].copy()
    if len(df_filtered) == 0:
        return None

    x = df_filtered['barrage_rate'].to_numpy(dtype=float)
    y = df_filtered[fr_column].to_numpy(dtype=float)

    mask = ~(np.isnan(x) | np.isnan(y))
    x_clean = x[mask]
    y_clean = y[mask]
    if len(x_clean) < 3:
        return None

    # --- z-score y within this file+region+celltype (i.e., within this column) ---
    y_z = zscore_within_vector(y_clean)
    # If zscore failed (std==0), bail
    if np.all(np.isnan(y_z)):
        return None

    # --- bin edges from x_clean ---
    if method == 'equal_count':
        # force sextiles by percentiles
        percentiles = np.linspace(0, 100, n_tiles + 1)
        tile_edges = np.percentile(x_clean, percentiles)

        # Important: if barrage_rate has many ties, percentiles can repeat -> empty bins.
        # We'll still handle via NaNs/0 counts below.
    elif method == 'equal_width':
        tile_edges = np.linspace(np.min(x_clean), np.max(x_clean), n_tiles + 1)
    else:
        raise ValueError(f"Unknown method: {method}")

    tile_means_x = []
    tile_means_yz = []
    tile_counts = []

    for i in range(n_tiles):
        if i == n_tiles - 1:
            mask_tile = (x_clean >= tile_edges[i]) & (x_clean <= tile_edges[i+1])
        else:
            mask_tile = (x_clean >= tile_edges[i]) & (x_clean < tile_edges[i+1])

        count = int(np.sum(mask_tile))
        tile_counts.append(count)

        if count > 0:
            tile_means_x.append(np.mean(x_clean[mask_tile]))
            tile_means_yz.append(np.mean(y_z[mask_tile]))
        else:
            tile_means_x.append(np.nan)
            tile_means_yz.append(np.nan)

    return {
        "tile_means_x": np.array(tile_means_x, float),
        "tile_means_y": np.array(tile_means_yz, float),  # z-scored means
        "tile_counts": np.array(tile_counts, int),
        "n_points": int(len(x_clean))
    }

# ==========================================
# COMPUTE PER-FILE STATS THEN POOL
# ==========================================
print("\n" + "="*60)
print(f"Computing Z-scored statistics using method: {BINNING_METHOD.upper()}...")
print("="*60)

pooled_data = {col: [] for col in fr_columns}

for idx, df in enumerate(all_dfs):
    filename = os.path.basename(pkl_files[idx])
    print(f"\nProcessing: {filename}")

    for col in fr_columns:
        stats = compute_binned_stats_zscore(df, col, n_tiles=n_tiles, method=BINNING_METHOD)
        if stats is not None:
            pooled_data[col].append(stats)
            print(f"  {col}: {stats['n_points']} points, tile counts: {stats['tile_counts']}")
        else:
            print(f"  {col}: skipped (missing/empty/zero-std)")

print("\n" + "="*60)
print("Computing pooled statistics (mean ± SEM across files per tile)...")
print("="*60)

pooled_stats = {}

for col in fr_columns:
    if len(pooled_data[col]) == 0:
        print(f"{col}: No data")
        continue

    all_tile_means_x = np.array([s["tile_means_x"] for s in pooled_data[col]], float)  # files x tiles
    all_tile_means_y = np.array([s["tile_means_y"] for s in pooled_data[col]], float)  # files x tiles (z)

    pooled_mean_x = np.nanmean(all_tile_means_x, axis=0)
    pooled_mean_y = np.nanmean(all_tile_means_y, axis=0)

    n_valid = np.sum(~np.isnan(all_tile_means_y), axis=0)
    pooled_std = np.nanstd(all_tile_means_y, axis=0)

    with np.errstate(divide='ignore', invalid='ignore'):
        pooled_sem = pooled_std / np.sqrt(n_valid)

    pooled_stats[col] = {
        "mean_x": pooled_mean_x,
        "mean_y": pooled_mean_y,
        "sem_y": pooled_sem,
        "n_files": len(pooled_data[col]),
        "n_valid": n_valid
    }
    print(f"{col}: pooled from {len(pooled_data[col])} files")

# ==========================================
# PLOTTING (Z-SCORED FR)
# ==========================================
# ==========================================
print("\n" + "="*60)
print("Creating pooled plots (z-scored FR, x = tile)...")
print("="*60)

In [ ]:
import matplotlib as mpl
import scipy.stats as spstats
# Global font: prefer Arial, with safe fallbacks
mpl.rcParams['svg.fonttype'] = 'none' 
mpl.rcParams['text.usetex'] = False
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = [
    'Helvetica Neue', 'Helvetica', 'Arial', 'Liberation Sans', 'DejaVu Sans'
]

# Mathtext to match
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Helvetica Neue'   # or 'Helvetica'
mpl.rcParams['mathtext.it'] = 'Helvetica Neue:italic'
mpl.rcParams['mathtext.bf'] = 'Helvetica Neue:bold'

# (optional) sizes + minus sign
mpl.rcParams['font.size'] = 16
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams['axes.unicode_minus'] = False

# Global sizes (tweak to taste)
mpl.rcParams['font.size'] = 14
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams["axes.linewidth"] = 1.5       # spine width
mpl.rcParams["xtick.major.width"] =1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5
mpl.rcParams["axes.linewidth"] = 1.5       # spine width
mpl.rcParams["xtick.major.width"] =1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5

col_name = "CA2_E_FR_per_unit"
title = "CA2 Excitatory"

fig, ax = plt.subplots(figsize=(6, 5))

st = pooled_stats[col_name]

tile_ids = np.arange(1, n_tiles + 1)
valid = np.isfinite(st["mean_y"])

x_plot = tile_ids[valid]
y_plot = st["mean_y"][valid]
sem_plot = st["sem_y"][valid]

ax.errorbar(
    x_plot, y_plot, yerr=sem_plot,
    fmt='o-', linewidth=2.5, markersize=10,
    capsize=6, capthick=2.5,label=None
)

ax.set_xlabel("Barrage rate sextile")
ax.set_ylabel("CA2 Pyr mean firing rate (z)")

ax.set_xticks(tile_ids)
ax.set_xticklabels([f"{i}" for i in tile_ids])

ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend()

plt.tight_layout()
save_path=r"../figure_output\S3_G_model_pooled_CA2_E_FR_per_unit_vs_barrage_tiles.svg"
plt.savefig(save_path, dpi=300)
plt.show()
w = 1.0 / np.clip(sem_plot.astype(float), 1e-12, np.inf)**2

# weighted linear fit: y = b0 + b1 x
b1, b0 = np.polyfit(x_plot.astype(float), y_plot.astype(float), deg=1, w=np.sqrt(w))

# get an approximate p-value for slope using weighted regression
# (using effective residual variance)
yhat = b0 + b1 * x_plot
resid = y_plot - yhat
dof = len(x_plot) - 2
s2 = np.sum(w * resid**2) / dof
Sxx = np.sum(w * (x_plot - np.average(x_plot, weights=w))**2)
se_b1 = np.sqrt(s2 / Sxx)
t = b1 / se_b1
p_slope = 2 * spstats.t.sf(np.abs(t), dof)

print(f"Weighted slope = {b1:.3f} ± {se_b1:.3f} (SE), t={t:.2f}, p={p_slope:.3g}, n={len(x_plot)}")
from scipy import stats
import numpy as np

# x_plot, y_plot already defined
r, p = stats.pearsonr(x_plot.astype(float), y_plot.astype(float))
print(f"Pearson r = {r:.3f}, p = {p:.3g}, n = {len(x_plot)}")

# optional: add to title
ax.set_title(f"{title} (Pearson r={r:.2f}, p={p:.2g})")